# The real hard work and uncapped patience and accuracy begins here because, We are now moving from EDA → Modeling, and we should be careful here because this is where leakage and unfair evaluation can easily happen.

There is one important methodological decision before we write the first cell:

Modeling design we will use

Because the model includes PreviousTurnout, RegisteredVotersChange, and RegistrationGrowth, the clean historical modelling population is the 717 wards that have a safe previous-election comparison.

That gives us:

Training: 2016
Held-out test: 2021
Baseline for 2021: previous turnout = 2016 turnout
Random Forest: trained on 2016, predicts 2021
Same 2021 test wards and same metrics for both models

We will not randomly split the panel.

Also, BaselinePredictedTurnout2026 will not be used to evaluate the historical model. It was constructed from 2021 turnout specifically as a 2026 baseline, so using it to predict 2021 would leak future information into the historical evaluation.

For the final prediction file, we can retain the historical baseline predictions where PreviousTurnout exists and put the Random Forest prediction only where the model was actually evaluated.

In [1]:
from pathlib import Path
import pandas as pd

# Use the validated Group 8 dataset as the only source for modelling.
# We keep the modelling inputs traceable to the feature-engineering stage.

base_dir = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed"
)

feature_path = (
    base_dir
    / "08_feature_engineered_dataset"
    / "ward_features_2011_2021.csv"
)

model_output_dir = (
    base_dir
    / "10_model_outputs"
)

model_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Feature file exists:", feature_path.exists())
print("Model output folder exists:", model_output_dir.exists())
print("Modelling source:", feature_path)
print("Model output folder:", model_output_dir)

Feature file exists: True
Model output folder exists: True
Modelling source: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\08_feature_engineered_dataset\ward_features_2011_2021.csv
Model output folder: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\10_model_outputs


In [2]:
# Before calculating the baseline, let's formally define the 2016 to 2021,
# forward test and verify how many wards are safely comparable across those periods.

import pandas as pd

# Load the feature-engineered dataset for the historical modelling test.
# We use only wards with a valid previous-turnout comparison for the 2016 → 2021 evaluation.

model_data = pd.read_csv(
    feature_path,
    encoding="utf-8-sig"
)

test_2021 = model_data[
    (model_data["ElectionYear"] == 2021)
    & model_data["PreviousTurnout"].notna()
    & model_data["BoundaryConsistent"]
].copy()

train_2016 = model_data[
    (model_data["ElectionYear"] == 2016)
    & model_data["PreviousTurnout"].notna()
    & model_data["BoundaryConsistent"]
].copy()

print("Full modelling dataset shape:", model_data.shape)

print("\nTraining period: 2016")
print("Training rows:", len(train_2016))

print("\nHeld-out test period: 2021")
print("Test rows:", len(test_2021))

print("\nUnique training wards:",
      train_2016["MunicipalityCode"].astype(str).str.strip()
      .str.cat(train_2016["Ward"].astype(str).str.strip(), sep="_")
      .nunique())

print("Unique test wards:",
      test_2021["MunicipalityCode"].astype(str).str.strip()
      .str.cat(test_2021["Ward"].astype(str).str.strip(), sep="_")
      .nunique())

Full modelling dataset shape: (2599, 53)

Training period: 2016
Training rows: 717

Held-out test period: 2021
Test rows: 717

Unique training wards: 717
Unique test wards: 717


In [3]:
# Use the previous election turnout as our naive prediction for the held-out 2021 election.
# This gives us a concrete benchmark that the Random Forest must improve on.

baseline_actual = test_2021["TurnoutRate"]
baseline_prediction = test_2021["PreviousTurnout"]

baseline_mae = (
    baseline_actual
    - baseline_prediction
).abs().mean()

baseline_rmse = (
    (
        baseline_actual
        - baseline_prediction
    ) ** 2
).mean() ** 0.5

print("Historical baseline: 2016 turnout -> 2021 turnout")
print("Test wards:", len(test_2021))
print("Baseline MAE:", round(baseline_mae, 2), "percentage points")
print("Baseline RMSE:", round(baseline_rmse, 2), "percentage points")

Historical baseline: 2016 turnout -> 2021 turnout
Test wards: 717
Baseline MAE: 11.64 percentage points
Baseline RMSE: 13.31 percentage points


so fromreviewing leaks we got:

2016 → 2021 naive baseline

717 test wards
MAE: 11.64 percentage points
RMSE: 13.31 percentage points

So the Random Forest needs to be evaluated on these same 717 wards, using the same target and metrics.

One methodological issue needs to be checked before we build the Random Forest: some of our contextual fields come from periods later than the election they are attached to. For example, the 2021 rows carry ProvinceContextYear = 2023, and the political-attitude data comes from 2024. Using those values to predict 2021 would introduce future information into the test.

We should audit that now rather than accidentally training a model with leakage.

==============================================================================================================================
==============================================================================================================================

Historical baseline benchmark

Using the previous election's turnout as the prediction gives us a baseline MAE of 11.64 percentage points and RMSE of 13.31 percentage points on 717 held-out 2021 wards. This establishes the error level that our Random Forest will be compared against using the same test population.

Before training the Random Forest, we must also ensure that every predictor represents information that could have been available at the time of the prediction. This is particularly important for our contextual datasets because some source periods occur after the historical election year. We therefore perform a temporal feature-availability audit before selecting the final modelling features.

In [4]:
# Check whether any contextual information comes from after the election being predicted.
# This keeps our historical test honest and prevents future information from entering the model.

future_context = model_data[
    model_data["ProvinceContextYear"] > model_data["ElectionYear"]
].copy()

print("Records with future ProvinceContextYear:",
      len(future_context))

print("\nFuture context by election year:")
print(
    future_context
    .groupby(
        ["ElectionYear", "ProvinceContextYear"]
    )
    .size()
)

print("\nPolitical-attitude coverage by election year:")
print(
    model_data
    .groupby("ElectionYear")[
        [
            "Dissatisfied with democracy",
            "Distrust of national government",
            "Distrust of political parties",
            "Distrust of provincial government",
            "Distrust of the IEC"
        ]
    ]
    .count()
)

Records with future ProvinceContextYear: 901

Future context by election year:
ElectionYear  ProvinceContextYear
2021          2023                   901
dtype: int64

Political-attitude coverage by election year:
              Dissatisfied with democracy  Distrust of national government  \
ElectionYear                                                                 
2011                                    0                                0   
2016                                    0                                0   
2021                                  901                              901   

              Distrust of political parties  \
ElectionYear                                  
2011                                      0   
2016                                      0   
2021                                    901   

              Distrust of provincial government  Distrust of the IEC  
ElectionYear                                                          
2011             

======================================================================================================================================================
======================================================================================================================================================

The auidit of prediction susgets:

Exactly. This is the leakage checkpoint we needed.

The result confirms that all 901 held-out 2021 rows carry ProvinceContextYear = 2023, which is information from after the 2021 election. The political-attitude variables are also only present in the 2021 rows, so including them would let the model use information that would not have been available when making a genuine 2021 forward prediction.

=======================================================================================================================================================
=======================================================================================================================================================

Temporal leakage decision

Our audit identified future information in the historical evaluation data. The 2021 records use a ProvinceContextYear of 2023, and the political-attitude indicators are populated only for the 2021 records. Using these variables in the 2016 → 2021 model would introduce information that occurred after the target election.

We therefore exclude these future-context variables from the historical Random Forest evaluation. This keeps the test representative of a genuine forward prediction and makes the comparison with our 11.64 percentage-point baseline MAE fair.

We retain the variables in the engineered dataset because they may still be useful for a later prediction scenario when the relevant information is available, but they will not be allowed to influence this historical benchmark.

In [5]:
#For now, we'll use the turnout-history features plus the poverty variables from the dataset, 
#but we need to make sure the 2021 test does not use its 2023 context. 
#The safest way is to create a historical context value from the latest context period #that was available before each election.

# Build the features that could have been available before each historical election.
# We remove future political-attitude information and use the latest prior province context.

target = "TurnoutRate"

turnout_features = [
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth",
    "ProvincialAverageTurnout",
    "BelowProvincialAverageTurnout"
]

poverty_features = [
    "Food poverty headcount (FPL, %)",
    "Gini coefficient (income per capita)",
    "Poverty gap (P1, %)",
    "Poverty headcount (P0, %)",
    "Poverty share (%)",
    "Severity of poverty (P2, %)"
]

political_attitude_features = [
    "Dissatisfied with democracy",
    "Distrust of national government",
    "Distrust of political parties",
    "Distrust of provincial government",
    "Distrust of the IEC"
]

print("Target:", target)
print("\nTurnout-history features:")
print(turnout_features)

print("\nContext features:")
print(poverty_features)

print("\nExcluded political-attitude features:")
print(political_attitude_features)

Target: TurnoutRate

Turnout-history features:
['PreviousTurnout', 'RegisteredVotersChange', 'RegistrationGrowth', 'ProvincialAverageTurnout', 'BelowProvincialAverageTurnout']

Context features:
['Food poverty headcount (FPL, %)', 'Gini coefficient (income per capita)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)']

Excluded political-attitude features:
['Dissatisfied with democracy', 'Distrust of national government', 'Distrust of political parties', 'Distrust of provincial government', 'Distrust of the IEC']


#The poverty context mapping is now temporally safe:

For 2016, we use the 2015 province context.
For 2021, we also use the latest context available before 2021, which is 2015.
We do not use the 2023 context for the 2021 test.

One more leakage issue remains with the two turnout-context features. ProvincialAverageTurnout for 2021 was calculated from 2021 turnout itself, and BelowProvincialAverageTurnout also depends on that same 2021 turnout. We must rebuild those two features using only information known before the target election.

=======================================***********************************************===========================================
=======================================***********************************************===========================================


Leakage-safe context preparation

We confirmed that the historical poverty context can be aligned without using information from after the target election. For the 2016 and 2021 modelling periods, we use the latest available province context before each election.

We also need to rebuild the provincial turnout benchmark features because their original 2021 values were calculated using 2021 turnout. Using those values in the 2021 test would give the model information derived from the target itself. We therefore replace them with benchmarks calculated from the preceding election period.


In [7]:
# Build the provincial turnout benchmark from the previous election only.
# This keeps the 2021 test independent from the actual 2021 turnout we are trying to predict.

provincial_turnout = (
    model_data
    .groupby("ElectionYear")["TurnoutRate"]
    .mean()
    .sort_index()
)

print("Observed provincial average turnout:")
print(provincial_turnout)

safe_turnout_context = []

for election_year in [2016, 2021]:
    previous_year = election_year - 5

    safe_turnout_context.append(
        {
            "ElectionYear": election_year,
            "SafeProvincialAverageTurnout": provincial_turnout.loc[
                previous_year
            ]
        }
    )

safe_turnout_context = pd.DataFrame(
    safe_turnout_context
)

print("\nLeakage-safe provincial turnout context:")
print(safe_turnout_context)

Observed provincial average turnout:
ElectionYear
2011    61.074435
2016    60.570788
2021    49.556774
Name: TurnoutRate, dtype: float64

Leakage-safe provincial turnout context:
   ElectionYear  SafeProvincialAverageTurnout
0          2016                     61.074435
1          2021                     60.570788


Good. That confirms the turnout benchmark is now leakage-safe:

2016 model: prior-election KZN average = 61.0744% from 2011.
2021 test: prior-election KZN average = 60.5708% from 2016.
We will not use the actual 2021 average of 49.5568% as a predictor

We rebuilt the provincial turnout benchmark so that each election uses only the previous election's information. For 2016, the benchmark is based on the 2011 KZN ward-level average, while for 2021 it is based on the 2016 average. This prevents the target year's turnout from entering the predictors and keeps the forward test consistent with how the model would be used in practice

Okay from here we will audit the demographic context for time leakage

Before we put the demographic variables into the Random Forest, we need to establish whether the municipality-context data changes across the three election years or is the same current snapshot repeated across historical rows.

In [9]:
# Define the demographic context fields used for the timing check.
# We need to confirm whether these values are historical or a repeated current snapshot.

demographic_fields = [
    "Household",
    "Homeless",
    "Transient",
    "Institution",
    "UrbanArea",
    "TribalOrTraditionalArea",
    "FarmArea",
    "MalePopulation",
    "Population"
]

demographic_timing = (
    model_data
    .groupby("ElectionYear")[demographic_fields]
    .nunique()
)

print("Unique demographic values by election year:")
print(demographic_timing)

print("\nDemographic rows available by election year:")
print(
    model_data
    .groupby("ElectionYear")[demographic_fields]
    .count()
)

Unique demographic values by election year:
              Household  Homeless  Transient  Institution  UrbanArea  \
ElectionYear                                                           
2011                 30         6         40           39         40   
2016                 33         7         44           43         44   
2021                 33         7         44           43         44   

              TribalOrTraditionalArea  FarmArea  MalePopulation  Population  
ElectionYear                                                                 
2011                               38        40              40          40  
2016                               42        44              44          44  
2021                               42        44              44          44  

Demographic rows available by election year:
              Household  Homeless  Transient  Institution  UrbanArea  \
ElectionYear                                                           
2011           

this result tells us something important before we train.

The demographic values do vary across election years in the merged dataset, but that alone does not prove the demographic source represents information that was available before each election. The counts are also affected by the different ward/municipality coverage in 2011, 2016 and 2021.

So we should make one final check: for a municipality that appears in multiple election years, are its demographic values actually changing, or are we simply seeing different coverage?

==================================================================================================================================================================================================================================================================================================================

In [10]:
# Compare demographic values for the same municipality across our election periods.
# This tells us whether the context is genuinely time-varying or repeated after the municipality merge.

demographic_change_check = (
    model_data
    .dropna(subset=["CurrentMunicipality"])
    .groupby("CurrentMunicipality")[demographic_fields]
    .nunique()
)

print("Municipalities with changing demographic values:")
print(
    (demographic_change_check > 1)
    .sum()
    .sort_values(ascending=False)
)

print("\nMunicipalities with unchanged demographic values:")
print(
    (demographic_change_check == 1)
    .sum()
    .sort_values(ascending=False)
)

print("\nTotal municipalities checked:",
      len(demographic_change_check))

Municipalities with changing demographic values:
Household                  0
Homeless                   0
Transient                  0
Institution                0
UrbanArea                  0
TribalOrTraditionalArea    0
FarmArea                   0
MalePopulation             0
Population                 0
dtype: int64

Municipalities with unchanged demographic values:
Household                  44
Homeless                   44
Transient                  44
Institution                44
UrbanArea                  44
TribalOrTraditionalArea    44
FarmArea                   44
MalePopulation             44
Population                 44
dtype: int64

Total municipalities checked: 44


Okay so assessing feature timing has confirmed the the demographic context is static municipality-level context repeated across election years.

That means we should not use it in the historical 2016 → 2021 test until we know it was available before 2021. Otherwise we risk introducing future demographic information into the test. We can still retain these fields for the eventual 2026 prediction model, where the current context is appropriate.

This also means we should separate our modelling into:

Historical benchmark model (2016 → 2021):
uses information that is safely available from the historical dataset before 2021.

2026 prediction model later:
can use the current municipality/demographic context and other 2026-available information.  





Interpretation:
The demographic values are unchanged for each municipality across all election years, confirming that they represent a static municipality-level context rather than year-specific demographic measurements.

Because the feature-engineered dataset does not identify the collection year of these demographic values, we will not assume that they were available before the 2021 election. To protect the historical evaluation from temporal leakage, these demographic features are excluded from the 2016 → 2021 benchmark model.

They remain available for the later 2026 prediction stage, where current municipality context can legitimately contribute to the prediction.

In [12]:
# Define the features that can safely be used for the historical 2016 → 2021 test.
# We keep the model focused on information available before the held-out election.

historical_features = [
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth",
    "SafeProvincialAverageTurnout",
    "BelowProvincialAverageTurnout",
    "Food poverty headcount (FPL, %)",
    "Gini coefficient (income per capita)",
    "Poverty gap (P1, %)",
    "Poverty headcount (P0, %)",
    "Poverty share (%)",
    "Severity of poverty (P2, %)"
]

print("Historical modelling features:")
for feature in historical_features:
    print("-", feature)

print("\nFeatures excluded from historical test:")
for feature in demographic_fields:
    print("-", feature)

Historical modelling features:
- PreviousTurnout
- RegisteredVotersChange
- RegistrationGrowth
- SafeProvincialAverageTurnout
- BelowProvincialAverageTurnout
- Food poverty headcount (FPL, %)
- Gini coefficient (income per capita)
- Poverty gap (P1, %)
- Poverty headcount (P0, %)
- Poverty share (%)
- Severity of poverty (P2, %)

Features excluded from historical test:
- Household
- Homeless
- Transient
- Institution
- UrbanArea
- TribalOrTraditionalArea
- FarmArea
- MalePopulation
- Population


The feature list is now fixed for the historical 2016 → 2021 test.

One correction we should make before building the matrix: BelowProvincialAverageTurnout from Group 8 is also target-year-derived. For a fair forward test, we need a safe version based on the previous election:

PreviousTurnout < SafeProvincialAverageTurnout

That uses only information known before the prediction.

In [13]:
# Attach the safe historical context to the 2016 and 2021 modelling rows.
# We also rebuild the below-average flag from the previous election so the target year is never used.

historical_model = model_data[
    model_data["ElectionYear"].isin([2016, 2021])
    & model_data["PreviousTurnout"].notna()
    & model_data["BoundaryConsistent"]
].copy()

historical_model = historical_model.merge(
    safe_context,
    on="ElectionYear",
    how="left",
    validate="many_to_one"
)

historical_model = historical_model.merge(
    safe_turnout_context,
    on="ElectionYear",
    how="left",
    validate="many_to_one"
)

historical_model["SafeBelowProvincialAverageTurnout"] = (
    historical_model["PreviousTurnout"]
    < historical_model["SafeProvincialAverageTurnout"]
)

historical_model["SafeBelowProvincialAverageTurnout"] = (
    historical_model["SafeBelowProvincialAverageTurnout"]
    .astype(int)
)

print("Historical modelling data shape:",
      historical_model.shape)

print("\nRows by election year:")
print(
    historical_model["ElectionYear"]
    .value_counts()
    .sort_index()
)

print("\nSafe context values:")
print(
    historical_model[
        [
            "ElectionYear",
            "SafeProvincialAverageTurnout",
            "SafeBelowProvincialAverageTurnout"
        ]
    ]
    .drop_duplicates()
    .sort_values("ElectionYear")
)

Historical modelling data shape: (1434, 61)

Rows by election year:
ElectionYear
2016    717
2021    717
Name: count, dtype: int64

Safe context values:
    ElectionYear  SafeProvincialAverageTurnout  \
0           2016                     61.074435   
2           2016                     61.074435   
1           2021                     60.570788   
11          2021                     60.570788   

    SafeBelowProvincialAverageTurnout  
0                                   1  
2                                   0  
1                                   0  
11                                  1  


The historical modelling dataset is now 1,434 rows: 717 for 2016 and 717 for 2021.

The two unexpected-looking rows for each year in the SafeBelowProvincialAverageTurnout output are not duplicates in the dataset; they are simply showing different wards with different flag values. That's correct because the flag is calculated per ward from PreviousTurnout.


interpretation:
Historical modelling population

We have established a forward historical modelling population of 717 comparable wards in 2016 and the same 717 wards in 2021. Wards with missing PreviousTurnout or a boundary-inconsistent historical comparison are excluded because a reliable forward comparison cannot be established for them.

The provincial turnout benchmark and below-average indicator have also been rebuilt using only information available before each target election. This prevents the 2021 target turnout from entering the predictors.

In [15]:
# Use the leakage-safe poverty values created from the prior context period.
# We restore their original names so the final modelling feature list stays clean.

for feature in poverty_features:
    safe_column = f"{feature}_y"

    if safe_column in historical_model.columns:
        historical_model[feature] = historical_model[safe_column]

model_features = [
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth",
    "SafeProvincialAverageTurnout",
    "SafeBelowProvincialAverageTurnout",
    "Food poverty headcount (FPL, %)",
    "Gini coefficient (income per capita)",
    "Poverty gap (P1, %)",
    "Poverty headcount (P0, %)",
    "Poverty share (%)",
    "Severity of poverty (P2, %)"
]

feature_check = historical_model[model_features].isna().sum()

print("Missing values in modelling features:")
print(feature_check)

print("\nNon-numeric modelling features:")
print(
    historical_model[model_features]
    .select_dtypes(exclude="number")
    .columns.tolist()
)

print("\nHistorical modelling feature check:",
      "PASSED"
      if (
          feature_check.sum() == 0
          and len(
              historical_model[model_features]
              .select_dtypes(exclude="number")
              .columns
          ) == 0
      )
      else "REVIEW REQUIRED")

Missing values in modelling features:
PreviousTurnout                         0
RegisteredVotersChange                  0
RegistrationGrowth                      0
SafeProvincialAverageTurnout            0
SafeBelowProvincialAverageTurnout       0
Food poverty headcount (FPL, %)         0
Gini coefficient (income per capita)    0
Poverty gap (P1, %)                     0
Poverty headcount (P0, %)               0
Poverty share (%)                       0
Severity of poverty (P2, %)             0
dtype: int64

Non-numeric modelling features:
[]

Historical modelling feature check: PASSED


================================================================================================================================
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

Okay now lets move on to Machine learning and testing:

Train/test split

We train on 2016 (predicting 2016 turnout using safe features derived from the 2011→2016 period) and test on 2021 (predicting 2021 turnout using safe features derived from the 2016→2021 period). This keeps the same forward-in-time structure as the leakage-safe features themselves — we're never using information from the target year to predict that same year.

The target column should be the ward-level turnout rate built in Group 1. If your column isn't named TurnoutRate, swap the name below before running.

In [17]:
# Split into a time-respecting train/test set.
# Train = 2016 (predicting 2016 turnout using pre-2016 information)
# Test  = 2021 (predicting 2021 turnout using pre-2021 information)

target_column = "TurnoutRate"   # <-- change this if your turnout column has a different name

train_data = historical_model[historical_model["ElectionYear"] == 2016].copy()
test_data  = historical_model[historical_model["ElectionYear"] == 2021].copy()

X_train = train_data[model_features]
y_train = train_data[target_column]

X_test = test_data[model_features]
y_test = test_data[target_column]

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("\nTarget summary (train):")
print(y_train.describe())
print("\nTarget summary (test):")
print(y_test.describe())

Train shape: (717, 11) | Test shape: (717, 11)

Target summary (train):
count    717.000000
mean      60.383007
std        5.710238
min       36.757624
25%       57.048640
50%       60.311804
75%       63.998977
max       78.779681
Name: TurnoutRate, dtype: float64

Target summary (test):
count    717.000000
mean      48.967049
std        7.899633
min       16.877470
25%       44.123314
50%       49.927571
75%       54.295455
max       73.341864
Name: TurnoutRate, dtype: float64


In [19]:
# Baseline = PreviousTurnout used directly as the prediction.
# This is the naive model described in the solution statement.

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

baseline_predictions = X_test["PreviousTurnout"]

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

print("Baseline (PreviousTurnout) performance on 2021 test set:")
print(f"MAE:  {baseline_mae:.3f} percentage points")
print(f"RMSE: {baseline_rmse:.3f} percentage points")

Baseline (PreviousTurnout) performance on 2021 test set:
MAE:  11.640 percentage points
RMSE: 13.309 percentage points


Train the Random Forest

Now we train the Random Forest regression model on the 2016 training rows, using the 11 leakage-safe features, and generate predictions for the 2021 test rows. We fix random_state for reproducibility. This first run uses reasonable default settings — we're checking whether the model beats the baseline at all before tuning anything.

In [20]:
# Train Random Forest on 2016, predict on 2021.
# Fixed random_state for reproducibility.

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

print("Random Forest performance on 2021 test set:")
print(f"MAE:  {rf_mae:.3f} percentage points")
print(f"RMSE: {rf_rmse:.3f} percentage points")

print("\nComparison to baseline:")
print(f"Baseline MAE:      {baseline_mae:.3f}")
print(f"Random Forest MAE: {rf_mae:.3f}")
print(f"Improvement:       {baseline_mae - rf_mae:.3f} percentage points "
      f"({100 * (baseline_mae - rf_mae) / baseline_mae:.1f}% reduction)")

Random Forest performance on 2021 test set:
MAE:  11.096 percentage points
RMSE: 12.968 percentage points

Comparison to baseline:
Baseline MAE:      11.640
Random Forest MAE: 11.096
Improvement:       0.544 percentage points (4.7% reduction)


^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>><<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

Evaluating model performance and planning next diagnostic steps.
Evaluating model performance and planning next diagnostic steps.
A real improvement, but a modest one — 4.7% reduction in MAE, and RMSE also dropped slightly (13.31 → 12.97). This is an honest result, not a dramatic one: the Random Forest edges out the baseline but doesn't blow past it. That's worth stating plainly in your write-up rather than oversold — it tells you the extra features (poverty, registration change, growth) are adding some signal beyond "last time's turnout," but the province-wide decline is still hard to fully explain with what's currently in the feature set. That's a legitimate finding, not a failure.

Next useful step: see which features the model actually leaned on — this feeds directly into the dashboard requirement to "show the main measurable factors used by the model."



Interpretation:

We inspect which features the Random Forest relied on most. This tells us whether the model is mainly driven by the ward's own recent history (PreviousTurnout, RegistrationGrowth) or genuinely using the socioeconomic context (poverty, Gini) to explain differences between wards. This result will also inform what gets surfaced on the dashboard per prediction.

In [21]:
# Inspect which features the Random Forest actually relied on.

importance_df = pd.DataFrame({
    "Feature": model_features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("Feature importance (Random Forest):")
print(importance_df)

Feature importance (Random Forest):
                                 Feature  Importance
0                        PreviousTurnout    0.608807
1                 RegisteredVotersChange    0.195726
2                     RegistrationGrowth    0.193765
3      SafeBelowProvincialAverageTurnout    0.001701
4           SafeProvincialAverageTurnout    0.000000
5        Food poverty headcount (FPL, %)    0.000000
6   Gini coefficient (income per capita)    0.000000
7                    Poverty gap (P1, %)    0.000000
8              Poverty headcount (P0, %)    0.000000
9                      Poverty share (%)    0.000000
10           Severity of poverty (P2, %)    0.000000


666666666666666666666666666666666666666666666666666666666666666666999999999999999999999999999999999999999999999999999999999999999999999999999999999999999




Well porvety does matter its just not a very relevant case on this problem and we trying to addess otherwise this is a very clean and crucial resul, but before treating it 
as "poverty doesnt matter" we need to or lest just check it


Investigation:
Why the socioeconomic features show zero importance

SafeProvincialAverageTurnout and all five poverty/Gini columns came from Group 2, the province-level context panel. Since this project covers KZN only, every ward in the same year shares the exact same province-level value for these columns — there is no ward-to-ward variation within a single year for the model to split on. A Random Forest can only find importance in a feature if it varies across the rows it's trying to tell apart. If poverty, Gini, and provincial turnout are the same number for all 717 wards in 2016, the model correctly finds them useless for distinguishing one ward from another, even though they're real and meaningful at the provincial level.

This is not a sign the features are wrong — it's a sign they're at the wrong geographic grain for a ward-level model. The three features that do vary ward to ward (PreviousTurnout, RegisteredVotersChange, RegistrationGrowth) are the only ones carrying weight, which is exactly what we'd expect.

Let's confirm this diagnosis directly before deciding what to do about it.



In [22]:
# Confirm whether the poverty/Gini/provincial features actually vary across wards
# within a single election year, or are constant (which would explain zero importance).

province_level_features = [
    "SafeProvincialAverageTurnout",
    "Food poverty headcount (FPL, %)",
    "Gini coefficient (income per capita)",
    "Poverty gap (P1, %)",
    "Poverty headcount (P0, %)",
    "Poverty share (%)",
    "Severity of poverty (P2, %)",
]

print("Number of unique values per feature, within each ElectionYear:\n")
print(
    historical_model
    .groupby("ElectionYear")[province_level_features]
    .nunique()
)

Number of unique values per feature, within each ElectionYear:

              SafeProvincialAverageTurnout  Food poverty headcount (FPL, %)  \
ElectionYear                                                                  
2016                                     1                                1   
2021                                     1                                1   

              Gini coefficient (income per capita)  Poverty gap (P1, %)  \
ElectionYear                                                              
2016                                             1                    1   
2021                                             1                    1   

              Poverty headcount (P0, %)  Poverty share (%)  \
ElectionYear                                                 
2016                                  1                  1   
2021                                  1                  1   

              Severity of poverty (P2, %)  
ElectionYear             

<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

Confirmed — every one of these seven columns is a single constant value per year across all 717 wards. Diagnosis proven, not assumed.

What this means and what we do about it

This confirms the province-level features (poverty, Gini, provincial turnout) cannot help the model distinguish one ward from another, because they don't vary within a year — they can only ever explain year-to-year differences, not ward-to-ward differences. With only two years in this training setup, that's not enough variation for the model to use them meaningfully either way.

This is a genuine, documentable limitation of the current feature set, directly tied to the ward-vs-province grain mismatch flagged all the way back in the Group 2 and Group 3 folder decisions. It is not something to fix by dropping the columns quietly — it should be written up plainly: the model currently explains turnout using ward-level election history only; province-level socioeconomic context was included but could not contribute at ward level with the available data.

Three honest paths forward, not mutually exclusive:

Stop here for this stage — keep the model as is, document the limitation clearly, and note that ward-level (not province-level) poverty/service-delivery data would be needed to test that hypothesis properly. This is a legitimate, defensible outcome for a qualifier submission.
Bring in Group 3's municipality-level context (population demographics, and whatever survived the Group 3 duplication check) — since municipality is a finer grain than province, those features would vary more within a year and could carry real signal, if they're numeric and complete.
Try Group 4's ward-level economic data, if it was ever confirmed genuinely ward-grain — this would be the only way to get true ward-to-ward socioeconomic variation.

Given during our 1st phase of Data merging and preperation Group 4 was excluded earlier as "not truly ward-level," option 2 is the most realistic next move if we want to try improving on the current result. Otherwise, option 1 is a clean, honest place to close out modeling for now.

Which direction do we want to take, try folding in the Group 3 municipality context to see if it adds real ward-level variation, or stop here and move to documenting/finalizing this result?

we cant stop here the Evaluating how much the model's insight actually improved. 

So bringin in Group 3: **Markdown — decision writeup**

> **Decision: bringing in municipality-level context**
>
> After confirming that our province-level features (poverty, Gini, provincial turnout) showed zero importance because they don't vary across wards within a year, we decided this was a variance problem, not a relevance problem. Province-level data is real and meaningful, but at that grain every ward in KZN shares the exact same value, so the model has nothing to distinguish one ward from another.
>
> We considered three options: stopping here and documenting the limitation, bringing in Group 3's municipality-level context, or revisiting Group 4's ward-level economic data. We ruled out Group 4 because we had already excluded it earlier for not being genuinely ward-level, and reopening that decision now would undo a judgement call we made carefully rather than improve the model honestly.
>
> We chose to bring in Group 3's municipality-level context instead, since municipality is a real step down in grain from province. KZN has roughly 54 municipalities rather than one province, so a municipality-level feature will actually differ from ward to ward depending on which municipality that ward sits in, unlike the province-level features we just tested.
>
> We see this as a meaningful test either way. If a municipality-level feature shows real importance next to PreviousTurnout, that tells us turnout is partly explained by who lives where, not just by a ward's own voting history. If it still shows close to zero importance even at this finer grain, that's also a legitimate and reportable finding, it would mean ward-level turnout in this dataset is currently best explained by election history alone, not by the demographic context available to us. Either outcome is worth having, and right now we don't have either, because we were testing at a grain where the answer was guaranteed to be zero.





00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

Bringing in municipality-level context

We now attach Group 3's municipality-level context (municipality_context.csv) to historical_model, joined on the municipality that each ward belongs to. Municipality-level data is time-invariant in our current dataset (a single snapshot, not a year panel), so the same municipality value will apply across both 2016 and 2021 — this is a documented limitation, the same kind we already accepted for demographic context in Group 3 itself, not a new one we're introducing here.

Before writing the merge, we need to confirm the exact column names on both sides.

In [23]:
# Check what we're working with before merging municipality context in.
# We need to confirm the exact ward-to-municipality key on both sides.

print("Columns in historical_model:")
print(historical_model.columns.tolist())

municipality_context = pd.read_csv(
    "data/processed/03_municipality_context/municipality_context.csv"
)

print("\nColumns in municipality_context:")
print(municipality_context.columns.tolist())

print("\nSample rows from municipality_context:")
print(municipality_context.head())

Columns in historical_model:
['Province', 'Municipality', 'Ward', 'RegisteredVoters', 'SpoiltVotes', 'TotalValidVotes', 'VotingDistricts', 'VotingStations', 'ElectionYear', 'TurnoutRate', 'BoundaryConsistent', 'MunicipalityCode', 'CurrentMunicipality', 'Household', 'Homeless', 'Transient', 'Institution', 'UrbanArea', 'TribalOrTraditionalArea', 'FarmArea', 'MalePopulation', 'Population', 'ProvinceContextYear', 'Food poverty headcount (FPL, %)_x', 'Gini coefficient (income per capita)_x', 'Poverty gap (P1, %)_x', 'Poverty headcount (P0, %)_x', 'Poverty share (%)_x', 'Severity of poverty (P2, %)_x', 'Dissatisfied with democracy', 'Distrust of national government', 'Distrust of political parties', 'Distrust of provincial government', 'Distrust of the IEC', 'QLFS_Q1_2024_Snapshot', 'qlfs_geography', 'QLFS_Q1_2024_DiscouragedWorkSeekers', 'QLFS_Q1_2024_Employed_thousand', 'QLFS_Q1_2024_AbsorptionRate', 'QLFS_Q1_2024_LabourForce_thousand', 'QLFS_Q1_2024_LabourForceParticipationRate', 'QLFS_Q1

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/03_municipality_context/municipality_context.csv'